<a href="https://colab.research.google.com/github/jennifersolano308-cyber/AAI2025/blob/main/Part_3_Self_Reflection_Prompt_for_Improving_Output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install openai

In [2]:
import os, re, time
from openai import OpenAI

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

client = OpenAI(api_key=os.environ["GEMINI_API_KEY"],
                base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
MODEL = "gemini-3.8-flash"

def pick_working_model():
    global MODEL
    try:
        client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=5)
        print("Using model:", MODEL)
    except Exception as e:
        if "404" in str(e) or "NOT_FOUND" in str(e):
            ids = [m.id.replace("models/", "") for m in client.models.list()]
            flash = [i for i in ids if "flash" in i and "image" not in i and "tts" not in i]
            if not flash:
                raise RuntimeError(f"No flash model found. Available: {ids}")
            MODEL = sorted(flash)[-1]
            print("Switched to model:", MODEL)
        else:
            raise

pick_working_model()

def call_llm(system, user, temperature=0.3, retries=3):
    for attempt in range(retries):
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=temperature,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}])
            return r.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e) and attempt < retries - 1:
                print("Rate limited, waiting 20s...")
                time.sleep(20)
            else:
                raise

Using model: gemini-3.8-flash


In [3]:
SOURCE = """Brightline Software, a company of 240 employees, ran a six-month pilot of a four-day work week
(Monday to Thursday, 32 paid hours) for its 60-person customer support department. Management expected
service quality to drop. Instead, average customer satisfaction scores rose from 4.1 to 4.4 out of 5, and
average first-response time stayed within 5 percent of the previous level. Employee burnout, measured by a
quarterly survey, fell from 38 percent of staff reporting high burnout to 21 percent. Voluntary resignations
in the department dropped to 2 during the pilot, compared with 7 in the previous six months.

The pilot did have costs. Because the support line had to stay open on Fridays, the company hired four
part-time agents to cover them, adding roughly $96,000 in labor costs over six months. Some employees also
reported that Thursdays felt rushed, and two team leads said scheduling meetings across the shortened week
was difficult. The company estimated that avoided recruiting and training costs from lower turnover offset
about $70,000 of the added expense. Leadership has proposed extending the pilot for another six months
and including the sales team, while tracking Friday coverage costs more closely."""

In [4]:
BASIC_SYSTEM = "You are a helpful assistant."
BASIC_USER = f"Summarize the following text.\n\n{SOURCE}"

original_summary = call_llm(BASIC_SYSTEM, BASIC_USER)
print("ORIGINAL SUMMARY (before):\n")
print(original_summary)

ORIGINAL SUMMARY (before):

**Summary:**

Brightline Software conducted a six-month, four-day workweek pilot for its 60-person customer support department. Rather than declining, customer satisfaction improved from 4.1 to 4.4 out of 5, response times remained steady, high employee burnout fell from 38% to 21%, and voluntary resignations dropped from 7 to 2. 

The trial presented some challenges, including rushed Thursdays, meeting scheduling difficulties, and $96,000 in additional labor costs to hire four part-time staff for Friday coverage—though lower turnover offset about $70,000 of that expense. Based on the results, leadership proposed extending the pilot for six months, expanding it to the sales team, and monitoring Friday coverage expenses more closely.


In [5]:
REQUIREMENTS = """1. ACCURACY: Only facts stated in the source. No invented numbers or claims. Numbers must match the source.
2. CLARITY: Plain language, no jargon, no vague phrases.
3. AUDIENCE: A busy executive deciding whether to adopt a four-day week.
4. LENGTH: 60 words maximum in total.
5. FORMAT: Exactly 3 bullet points, each 20 words or fewer, and no other text (no intro, no conclusion).
6. FOCUS: Bullet 1 = results, Bullet 2 = trade-offs/costs, Bullet 3 = recommended next step."""

CRITIQUE_SYSTEM = """You are a strict editor who reviews summaries against a checklist.
For EACH of the six requirements, output one line in this form:
<REQUIREMENT NAME>: PASS or FAIL - one sentence of specific evidence (quote or count the offending text).
Then output a section "ISSUES TO FIX" with a numbered list of concrete changes needed.
Rules: be specific, do not rewrite the summary yourself, do not praise; check every number against the source."""

critique_input = f"""SOURCE TEXT:
{SOURCE}

REQUIREMENTS:
{REQUIREMENTS}

SUMMARY TO REVIEW:
{original_summary}"""

critique = call_llm(CRITIQUE_SYSTEM, critique_input, temperature=0.1)
print("SELF-CRITIQUE:\n")
print(critique)

SELF-CRITIQUE:

ACCURACY: PASS - All reported metrics, including satisfaction scores (4.1 to 4.4), burnout rates (38% to 21%), resignations (7 to 2), and expenses ($96,000 offset by $70,000), match the source text.
CLARITY: PASS - The text uses direct, accessible language with clear metrics and no confusing terminology.
AUDIENCE: PASS - The summary targets executive decision-makers by focusing on customer satisfaction, retention, net financial costs, and strategic plans.
LENGTH: FAIL - The summary contains 107 words, exceeding the 60-word maximum by 47 words.
FORMAT: FAIL - The text contains an introductory header ("**Summary:**") and two paragraphs rather than exactly three bullet points of 20 words or fewer.
FOCUS: FAIL - The content is organized as two narrative blocks rather than three distinct bullet points dedicated respectively to results, trade-offs/costs, and next steps.

ISSUES TO FIX
1. Remove the introductory title "**Summary:**" entirely so that no non-bullet text remains.

In [7]:
def call_llm(system, user, temperature=0.3, retries=6):
    global MODEL
    fallbacks = []
    try:
        ids = [m.id.replace("models/", "") for m in client.models.list()]
        fallbacks = [i for i in ids if "flash" in i and i != MODEL
                     and "image" not in i and "tts" not in i][:3]
    except Exception:
        pass

    for attempt in range(retries):
        try:
            r = client.chat.completions.create(
                model=MODEL, temperature=temperature,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}])
            return r.choices[0].message.content.strip()
        except Exception as e:
            msg = str(e)
            temporary = any(c in msg for c in ["429", "503", "UNAVAILABLE", "500"])
            if not temporary or attempt == retries - 1:
                raise
            wait = min(10 * (2 ** attempt), 60)
            # after 2 failures, try a different model
            if attempt >= 2 and fallbacks:
                MODEL = fallbacks.pop(0)
                print(f"Switching to model: {MODEL}")
            print(f"Server busy (attempt {attempt + 1}/{retries}), waiting {wait}s...")
            time.sleep(wait)